# Sprint 4: Pipeline & Deployment
**Goal: Wrap the full workflow into one Pipeline object, then deploy as a Streamlit app**

| What | Detail |
|-----|--------|
| Dataset | AI Jobs dataset (Sprint 1) |
| Best model | Final tuned model from Sprint 3 |
| Pipeline | Raw CSV row → predicted salary in one call |
| App | Streamlit UI in VS Code |


# Imports

In [1]:
import pandas as pd
import numpy as np
import joblib
import json, os, warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

print('✅ Imports done')


✅ Imports done


---
# Step 1: Load All Sprint Artefacts
> We load everything saved across Sprint 1, 2, 3 — no retraining needed.


In [2]:
save_dir = '/Users/rezwanshehik/Downloads/'

# Sprint 1 artefacts
with open(save_dir + 'encoding_maps.json') as f:
    encoding_maps = json.load(f)
scaler_s1 = joblib.load(save_dir + 'scaler_sprint1.joblib')
with open(save_dir + 'feature_columns.json') as f:
    feature_columns = json.load(f)     # column order after Sprint 1 encoding

# Sprint 3 artefacts
final_model = joblib.load(save_dir + 'final_model_sprint3.joblib')
with open(save_dir + 'best_features_sprint3.json') as f:
    best_features = json.load(f)       # subset of features used by final model

print('✅ All artefacts loaded!')
print(f'   Encoding maps     : {list(encoding_maps.keys())}')
print(f'   Feature columns   : {len(feature_columns)} columns')
print(f'   Final model type  : {type(final_model).__name__}')
print(f'   Best features     : {best_features}')


✅ All artefacts loaded!
   Encoding maps     : ['exp_map', 'size_map', 'edu_map', 'freq_cols', 'freq_maps']
   Feature columns   : 20 columns
   Final model type  : GradientBoostingRegressor
   Best features     : ['job_title', 'salary_currency', 'experience_level', 'employment_type', 'company_location', 'company_size', 'employee_residence', 'remote_ratio', 'education_required', 'years_experience', 'industry', 'job_description_length', 'benefits_score', 'application_duration', 'posting_month', 'posting_year', 'posting_day', 'day_of_week', 'days_to_apply', 'skills_count']


---
# Step 2: Build Custom Preprocessing Transformer
> `sklearn` Pipeline needs all steps to be Transformer objects.
> We wrap our Sprint 1 preprocessing logic into one custom transformer.


In [3]:
class SalaryPreprocessor(BaseEstimator, TransformerMixin):
    """
    Replicates Sprint 1 preprocessing:
      1. Date feature extraction
      2. skills_count from required_skills
      3. Drop raw columns
      4. Ordinal encoding (experience_level, company_size, education_required)
      5. Frequency encoding (job_title, salary_currency, employment_type,
                             company_location, employee_residence, industry)
      6. Select & order final feature columns
    """
    def __init__(self, encoding_maps, feature_columns, best_features):
        self.encoding_maps  = encoding_maps
        self.feature_columns = feature_columns
        self.best_features  = best_features

    def fit(self, X, y=None):
        return self   # maps already learnt from Sprint 1 training data

    def transform(self, X):
        df = X.copy()

        # ── Date features ───────────────────────────────────────────
        for col in ['posting_date', 'application_deadline']:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors='coerce')

        if 'posting_date' in df.columns:
            df['posting_month']  = df['posting_date'].dt.month
            df['posting_year']   = df['posting_date'].dt.year
            df['posting_day']    = df['posting_date'].dt.day
            df['day_of_week']    = df['posting_date'].dt.dayofweek

        if 'posting_date' in df.columns and 'application_deadline' in df.columns:
            df['application_duration'] = (
                df['application_deadline'] - df['posting_date']
            ).dt.days
            df['days_to_apply'] = df['application_duration']

        # ── skills_count ────────────────────────────────────────────
        if 'required_skills' in df.columns:
            df['skills_count'] = df['required_skills'].apply(
                lambda x: len(str(x).split(',')))

        # ── Drop raw / id columns ───────────────────────────────────
        drop_cols = ['job_id','posting_date','application_deadline',
                     'required_skills','company_name','application_duration']
        df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

        # ── Ordinal encoding ────────────────────────────────────────
        maps = self.encoding_maps
        if 'experience_level' in df.columns:
            df['experience_level']   = df['experience_level'].map(maps['exp_map'])
        if 'company_size' in df.columns:
            df['company_size']       = df['company_size'].map(maps['size_map'])
        if 'education_required' in df.columns:
            df['education_required'] = df['education_required'].map(maps['edu_map'])

        # ── Frequency encoding ───────────────────────────────────────
        for col in maps['freq_cols']:
            if col in df.columns:
                freq_map = maps['freq_maps'].get(col, {})
                df[col] = df[col].map(freq_map).fillna(0)

        # ── Select columns in exact Sprint 1 order ───────────────────
        available = [c for c in self.feature_columns if c in df.columns]
        df = df[available]

        # ── Subset to best_features (Sprint 3 RFE selection) ─────────
        best_avail = [c for c in self.best_features if c in df.columns]
        if best_avail:
            df = df[best_avail]

        return df

print('✅ SalaryPreprocessor defined!')


✅ SalaryPreprocessor defined!


---
# Step 3: Build the Full Pipeline
```
Raw input row
   ↓  SalaryPreprocessor   (date features, encoding, feature selection)
   ↓  StandardScaler       (scale to zero mean, unit variance)
   ↓  Final Model          (tuned RF / GB from Sprint 3)
   → Predicted Salary (USD)
```


In [4]:
full_pipeline = Pipeline(steps=[
    ('preprocessor', SalaryPreprocessor(
                        encoding_maps   = encoding_maps,
                        feature_columns = feature_columns,
                        best_features   = best_features)),
    ('scaler',        StandardScaler()),
    ('model',         final_model)
])

print('✅ Full Pipeline built!')
print(full_pipeline)


✅ Full Pipeline built!
Pipeline(steps=[('preprocessor',
                 SalaryPreprocessor(best_features=['job_title',
                                                   'salary_currency',
                                                   'experience_level',
                                                   'employment_type',
                                                   'company_location',
                                                   'company_size',
                                                   'employee_residence',
                                                   'remote_ratio',
                                                   'education_required',
                                                   'years_experience',
                                                   'industry',
                                                   'job_description_length',
                                                   'benefits_score',
                                      

# Step 4: Fit Pipeline on Raw Training Data & Evaluate
> We load the RAW data (before any encoding) and let the pipeline handle everything.


In [5]:
# Load raw dataset
raw_df = pd.read_csv(save_dir.rstrip('/') + '/ai_job_dataset.csv') \
         if os.path.exists(save_dir + 'ai_job_dataset.csv') \
         else pd.read_csv('/Users/rezwanshehik/Downloads/ai_job_dataset.csv')

# Lowercase columns (Sprint 1 step)
raw_df.columns = raw_df.columns.str.lower().str.strip().str.replace(' ','_')

# Remove job_id if present
if 'job_id' in raw_df.columns:
    raw_df.drop(columns=['job_id'], inplace=True)

X_raw = raw_df.drop(columns=['salary_usd'])
y_raw = raw_df['salary_usd']

from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

print(f'✅ Raw data loaded: {X_raw.shape}')
print(f'   Train: {X_tr.shape}  |  Test: {X_te.shape}')


✅ Raw data loaded: (15000, 17)
   Train: (12000, 17)  |  Test: (3000, 17)


In [6]:
# Fit pipeline on raw training data
full_pipeline.fit(X_tr, y_tr)

# Evaluate on raw test data
y_pred = full_pipeline.predict(X_te)

r2   = r2_score(y_te, y_pred)
mae  = mean_absolute_error(y_te, y_pred)
rmse = np.sqrt(mean_squared_error(y_te, y_pred))

print('='*50)
print('  PIPELINE EVALUATION (Raw → Prediction)')
print('='*50)
print(f'  Test R²   : {r2:.4f}')
print(f'  Test RMSE : ${rmse:,.0f}')
print(f'  Test MAE  : ${mae:,.0f}')
print('='*50)


  PIPELINE EVALUATION (Raw → Prediction)
  Test R²   : 0.6339
  Test RMSE : $36,539
  Test MAE  : $27,212


# Step 5: Single Raw Row Prediction Test
> Simulates exactly what the Streamlit app will send.


In [7]:
sample_row = pd.DataFrame([{
    'job_title'          : 'Data Scientist',
    'experience_level'   : 'SE',
    'employment_type'    : 'FT',
    'company_size'       : 'M',
    'education_required' : 'Master',
    'remote_ratio'       : 50,
    'benefits_score'     : 75,
    'years_experience'   : 5,
    'job_description_length': 800,
    'posting_date'       : '2024-03-15',
    'application_deadline': '2024-04-15',
    'required_skills'    : 'Python, SQL, ML',
    'company_location'   : 'US',
    'employee_residence' : 'US',
    'salary_currency'    : 'USD',
    'industry'           : 'Technology',
    'company_name'       : 'TechCorp'
}])

predicted = full_pipeline.predict(sample_row)[0]
print(f'✅ Sample prediction:')
print(f'   Job     : Data Scientist, Senior, M-company, 5yr exp')
print(f'   Salary  : ${predicted:,.0f} / year')
print(f'   Monthly : ${predicted/12:,.0f} / month')


✅ Sample prediction:
   Job     : Data Scientist, Senior, M-company, 5yr exp
   Salary  : $119,629 / year
   Monthly : $9,969 / month


# Step 6: Save the Full Pipeline


In [8]:
pipeline_path = save_dir + 'salary_pipeline_sprint4.joblib'
joblib.dump(full_pipeline, pipeline_path)
print(f'✅ Full pipeline saved → {pipeline_path}')

# Verify
loaded_pipe = joblib.load(pipeline_path)
r2_verify   = r2_score(y_te, loaded_pipe.predict(X_te))
print(f'✅ Loaded pipeline Test R² = {r2_verify:.4f}  ← matches above')


✅ Full pipeline saved → /Users/rezwanshehik/Downloads/salary_pipeline_sprint4.joblib
✅ Loaded pipeline Test R² = 0.6339  ← matches above


---
# Step 7: Create Streamlit App (`app.py`)
> Run this cell to write `app.py` to your project folder.
> Then open VS Code terminal and run: `streamlit run app.py`


In [9]:
app_code = '''
import streamlit as st
import pandas as pd
import joblib
import json, os, datetime

# ── Page setup ────────────────────────────────────────────────────
st.set_page_config(page_title="AI Job Salary Predictor", page_icon="💼", layout="wide")

# ── Load pipeline ─────────────────────────────────────────────────
PIPELINE_PATH = os.path.join(os.path.dirname(__file__), "salary_pipeline_sprint4.joblib")

@st.cache_resource
def load_pipeline():
    return joblib.load(PIPELINE_PATH)

pipeline = load_pipeline()

# ── Header ────────────────────────────────────────────────────────
st.markdown("""
    <h1 style=\'text-align:center;color:#1a5276;\'>💼 AI Job Salary Predictor</h1>
    <p style=\'text-align:center;color:gray;\'>Sprint 4 · Innomatics Research Labs · AI Jobs Dataset</p>
    <hr>
""", unsafe_allow_html=True)

# ── Inputs ────────────────────────────────────────────────────────
st.subheader("📝 Enter Job Details")
col1, col2, col3 = st.columns(3)

with col1:
    job_title = st.selectbox("Job Title",
        ["Data Scientist","Machine Learning Engineer","Data Analyst",
         "AI Engineer","Data Engineer","Research Scientist","MLOps Engineer"])
    experience_level = st.selectbox("Experience Level",
        ["EN","MI","SE","EX"],
        format_func=lambda x: {"EN":"Entry","MI":"Mid","SE":"Senior","EX":"Executive"}[x])
    employment_type = st.selectbox("Employment Type",
        ["FT","PT","CT","FL"],
        format_func=lambda x: {"FT":"Full Time","PT":"Part Time","CT":"Contract","FL":"Freelance"}[x])
    years_experience = st.slider("Years of Experience", 0, 20, 3)

with col2:
    company_size = st.selectbox("Company Size",
        ["S","M","L"],
        format_func=lambda x: {"S":"Small","M":"Medium","L":"Large"}[x])
    education_required = st.selectbox("Education Required",
        ["Associate","Bachelor","Master","PhD"])
    remote_ratio = st.selectbox("Remote Ratio",
        [0, 50, 100],
        format_func=lambda x: {0:"On-site",50:"Hybrid",100:"Fully Remote"}[x])
    benefits_score = st.slider("Benefits Score", 0, 100, 70)

with col3:
    company_location  = st.selectbox("Company Location",
        ["US","GB","CA","DE","IN","AU","FR","SG"])
    employee_residence = st.selectbox("Employee Residence",
        ["US","GB","CA","DE","IN","AU","FR","SG"])
    industry = st.selectbox("Industry",
        ["Technology","Finance","Healthcare","Education","Retail","Manufacturing"])
    required_skills   = st.text_input("Required Skills (comma-separated)",
        value="Python, SQL, Machine Learning")

st.markdown("<br>", unsafe_allow_html=True)

# ── Predict ───────────────────────────────────────────────────────
if st.button("🔮  Predict Salary", use_container_width=True, type="primary"):

    today   = pd.Timestamp.today()
    dead    = today + pd.Timedelta(days=30)

    input_df = pd.DataFrame([{
        "job_title"              : job_title,
        "experience_level"       : experience_level,
        "employment_type"        : employment_type,
        "company_size"           : company_size,
        "education_required"     : education_required,
        "remote_ratio"           : remote_ratio,
        "benefits_score"         : benefits_score,
        "years_experience"       : years_experience,
        "job_description_length" : 800,
        "posting_date"           : str(today.date()),
        "application_deadline"   : str(dead.date()),
        "required_skills"        : required_skills,
        "company_location"       : company_location,
        "employee_residence"     : employee_residence,
        "salary_currency"        : "USD",
        "industry"               : industry,
        "company_name"           : "Company"
    }])

    prediction = pipeline.predict(input_df)[0]

    st.markdown("<hr>", unsafe_allow_html=True)
    st.markdown(f"""
    <div style=\'background:linear-gradient(135deg,#1a5276,#2980b9);
               border-radius:15px;padding:28px;text-align:center;\'>
        <h2 style=\'color:white;margin:0;\'>💰 Predicted Annual Salary</h2>
        <h1 style=\'font-size:3.2rem;color:#f9e79f;margin:10px 0;\'>${prediction:,.0f}</h1>
        <p style=\'color:#d6eaf8;font-size:1.1rem;margin:0;\'>≈ ${prediction/12:,.0f} / month</p>
    </div>
    """, unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)
    c1, c2, c3 = st.columns(3)
    c1.metric("💵 Annual",  f"${prediction:,.0f}")
    c2.metric("📅 Monthly", f"${prediction/12:,.0f}")
    c3.metric("📆 Weekly",  f"${prediction/52:,.0f}")

    # ── Log ───────────────────────────────────────────────────────
    log_path = os.path.join(os.path.dirname(__file__), "prediction_logs.jsonl")
    with open(log_path, "a") as f:
        import json as _j
        f.write(_j.dumps({
            "timestamp"     : str(datetime.datetime.now()),
            "job_title"     : job_title,
            "experience"    : experience_level,
            "years_exp"     : years_experience,
            "company_size"  : company_size,
            "prediction"    : round(float(prediction), 2)
        }) + "\\n")
    st.caption("✅ Prediction logged to prediction_logs.jsonl")

# ── Sidebar: history ──────────────────────────────────────────────
st.sidebar.title("📜 Prediction History")
log_path = os.path.join(os.path.dirname(__file__), "prediction_logs.jsonl")
if os.path.exists(log_path):
    import json as _j
    lines = open(log_path).readlines()[-10:]
    for line in reversed(lines):
        r = _j.loads(line)
        st.sidebar.markdown(
            f"**{r[\'job_title\']}** ({r[\'experience\']} · {r[\'years_exp\']}yr)  \\n"
            f"💰 **${r[\'prediction\']:,.0f}**  \\n"
            f"<small style=\'color:gray\'>{r[\'timestamp\'][:16]}</small>",
            unsafe_allow_html=True)
        st.sidebar.divider()
    if st.sidebar.button("🗑️ Clear History"):
        os.remove(log_path)
        st.rerun()
else:
    st.sidebar.info("No predictions yet.")

# ── Footer ────────────────────────────────────────────────────────
st.markdown("<hr><p style=\'text-align:center;color:gray;\'>Sprint 4 · Innomatics Research Labs</p>",
            unsafe_allow_html=True)
'''

app_path = save_dir + 'app.py'
with open(app_path, 'w') as f:
    f.write(app_code.strip())
print(f'✅ app.py written → {app_path}')
print()
print('  To run in VS Code terminal:')
print(f'  cd {save_dir}')
print('  streamlit run app.py')


✅ app.py written → /Users/rezwanshehik/Downloads/app.py

  To run in VS Code terminal:
  cd /Users/rezwanshehik/Downloads/
  streamlit run app.py


---
# Step 8: Run in VS Code
## 1. Open VS Code terminal (`Ctrl + \``)
## 2. Install Streamlit (only once):
```bash
pip install streamlit
```
## 3. Run the app:
```bash
cd /Users/rezwanshehik/Downloads
streamlit run app.py
```
## 4. Browser opens at `http://localhost:8501` ✅

---
## Files generated by all 4 Sprints:
| File | Created in |
|------|----------|
| `encoding_maps.json` | Sprint 1 |
| `scaler_sprint1.joblib` | Sprint 1 |
| `feature_columns.json` | Sprint 1 |
| `X_train_final.csv`, `X_test_final.csv` | Sprint 1 |
| `best_model_sprint2.joblib` | Sprint 2 |
| `final_model_sprint3.joblib` | Sprint 3 |
| `best_features_sprint3.json` | Sprint 3 |
| `salary_pipeline_sprint4.joblib` | Sprint 4 ← full pipeline |
| `app.py` | Sprint 4 ← Streamlit app |


In [10]:
print('='*55)
print('  ALL 4 SPRINTS COMPLETE 🎉')
print('='*55)
print('  Sprint 1 → EDA & Preprocessing')
print('  Sprint 2 → Model Building & Evaluation')
print('  Sprint 3 → Optimization & Final Model')
print('  Sprint 4 → Pipeline & Deployment')
print('='*55)
print(f'  Pipeline Test R² : {r2:.4f}')
print(f'  Pipeline Test MAE: ${mae:,.0f}')
print('='*55)
print('  → streamlit run app.py')
print('='*55)


  ALL 4 SPRINTS COMPLETE 🎉
  Sprint 1 → EDA & Preprocessing
  Sprint 2 → Model Building & Evaluation
  Sprint 3 → Optimization & Final Model
  Sprint 4 → Pipeline & Deployment
  Pipeline Test R² : 0.6339
  Pipeline Test MAE: $27,212
  → streamlit run app.py
